# 13 Final models and comparison

This notebook puts the time-series models from before next to XGBoost, LightGBM, a tuned Prophet, LSTM, GRU, PatchTST and
VAR, and then tests three ways of combining them. Everything is scored on the same 150 items and the same six test
windows of 28 days. The numbers come from saved files, nothing is typed in by hand. The heavy fits are run by
`src/run_models.py` and cached in `data/processed/final/`, so this notebook only loads them and is quick to rerun.

The level stays as before: one series per item (the four California stores added up), 28 days ahead. I did not change it to
get smaller numbers. At category level the errors are bigger in units but smaller in percent, so I show the category
totals too (VAR section) and say which level each number is for.

In [1]:
import sys, json, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import TABLES, FIGS, PROCESSED, EVAL_ORIGINS, ORIGINS, FIG_STYLE, SEED, make_dirs
import ts_models as tm
import features as F
import ml_models as M
import final_eval as FE
import final_plots as FP
import vol_models as VM
from evaluation import mase_scales

make_dirs()
plt.rcParams.update(FIG_STYLE)
written = []
VAL = tm.VAL_ORIGINS
inp = tm.load_inputs()
daily, items = inp["daily"], list(inp["series"])
print(len(items), "items,", len(inp["cal"]), "calendar days")

150 items, 1969 calendar days


## The split

Train, then three validation windows (a rolling-origin backtest with three folds), then six test windows. Anything that is
chosen (hyper-parameters, which models to average, stacking weights, which classical model feeds the hybrid) is chosen on
the validation windows only. The test windows are scored once, in the leaderboard further down.

In [2]:
day = pd.Timedelta(days=1)
split = pd.DataFrame([
    {"part": "train", "origins": "all history before the first validation origin", "first_day": inp["long_df"]["date"].min().date(),
     "last_day": VAL[0].date()},
    {"part": "validation fold 1", "origins": VAL[0].date(), "first_day": (VAL[0] + day).date(), "last_day": (VAL[0] + 28 * day).date()},
    {"part": "validation fold 2", "origins": VAL[1].date(), "first_day": (VAL[1] + day).date(), "last_day": (VAL[1] + 28 * day).date()},
    {"part": "validation fold 3", "origins": VAL[2].date(), "first_day": (VAL[2] + day).date(), "last_day": (VAL[2] + 28 * day).date()},
    {"part": "test (6 windows, scored once)", "origins": f"{EVAL_ORIGINS[0].date()} to {EVAL_ORIGINS[-1].date()}",
     "first_day": (EVAL_ORIGINS[0] + day).date(), "last_day": (EVAL_ORIGINS[-1] + 28 * day).date()}])
split.to_csv(TABLES / "final_split.csv", index=False)
written.append(TABLES / "final_split.csv")
split

,part,origins,first_day,last_day
0,train,all history before the first validation origin,2011-01-29,2015-09-13
1,validation fold 1,2015-09-13,2015-09-14,2015-10-11
2,validation fold 2,2015-10-11,2015-10-12,2015-11-08
3,validation fold 3,2015-11-08,2015-11-09,2015-12-06
4,"test (6 windows, scored once)",2015-12-06 to 2016-04-24,2015-12-07,2016-05-22


## Shared features

One feature table feeds XGBoost, LightGBM and the hybrid, and the neural nets use the same inputs as windows. It is a direct
set-up: one row for each item, forecast origin and step ahead (1 to 28). What looks at the past (rolling means over 7 to 112
days, spread, share of zero days, the same weekday 1 to 4 weeks back) only reads sales up to the origin. What describes the
target day (weekday, month, yearly sine and cosine, events and their type, SNAP, Christmas, relative price) is known ahead,
because the calendar and prices are set in advance. Trees need no scaling. The neural nets divide sales by the item's mean
of the last 56 days up to the origin and standardise the other inputs with numbers taken from the training windows only.

In [3]:
rows = M.get_rows(inp)
print(rows.shape, "rows,", len(F.FEATURES), "features, origins", rows["origin"].min().date(), "to", rows["origin"].max().date())
feat = pd.DataFrame({"feature": F.FEATURES, "group": ["past"] * len(F.PAST) + ["known ahead"] * len(F.CAL) + ["item"] * len(F.STATIC)})
feat.to_csv(TABLES / "final_features.csv", index=False)
written.append(TABLES / "final_features.csv")
feat.groupby("group")["feature"].apply(lambda s: ", ".join(s)).to_frame()

(974064, 38) rows, 34 features, origins 2011-05-22 to 2016-04-24


,feature
group,
item,"cat, dept"
known ahead,"h, dow, dom, month, sin1, cos1, sin2, cos2, sn..."
past,"mean7, mean14, mean28, mean56, mean112, std28,..."


### No future information in the features

The check: take an item and an origin, replace every sale after the origin with random numbers, build the features again, and
see if anything changed. If a feature leaked, it would change.

In [4]:
rng = np.random.default_rng(SEED)
checks = []
for it in rng.choice(items, 10, replace=False):
    o = tm.ALL_ORIGINS[int(rng.integers(0, len(tm.ALL_ORIGINS)))]
    d = inp["series"][it].copy()
    d2 = d.copy()
    m = d2["date"] > o
    d2.loc[m, "sales"] = rng.integers(0, 999, m.sum())
    a = F.build_rows({it: d}, inp["cal"], inp["prices"], [o]); a = a[a["origin"] == o]
    b = F.build_rows({it: d2}, inp["cal"], inp["prices"], [o]); b = b[b["origin"] == o]
    same = bool(np.allclose(a[F.FEATURES].to_numpy(), b[F.FEATURES].to_numpy(), equal_nan=True))
    checks.append({"item": it, "origin": o.date(), "features_unchanged": same, "target_changed": bool((a["y"].to_numpy() != b["y"].to_numpy()).any())})
checks = pd.DataFrame(checks)
assert checks["features_unchanged"].all()
checks

,item,origin,features_unchanged,target_changed
0,HOUSEHOLD_2_409,2016-01-03,True,True
1,HOUSEHOLD_1_096,2016-04-24,True,True
2,FOODS_3_007,2015-11-08,True,True
3,HOBBIES_1_404,2015-09-13,True,True
4,HOBBIES_1_148,2016-04-24,True,True
5,HOBBIES_1_130,2016-04-24,True,True
6,HOUSEHOLD_1_055,2016-01-31,True,True
7,FOODS_3_099,2015-09-13,True,True
8,FOODS_3_503,2016-02-28,True,True
9,HOUSEHOLD_1_372,2016-02-28,True,True


## Load every model

The earlier models come from the saved forecasts of 07h (already with the blow-up guard), the new ones from the cache. The same
guard (a forecast above three times the most the item sold in training is replaced by seasonal naive) is applied to the new
models too. The neural nets were run with three seeds; their rows are kept per seed, and the seed average is used only when
building combinations.

In [5]:
old = FE.load_old()
new, seeds, caught = FE.load_new(daily)
fc9 = pd.concat([old, new], ignore_index=True)
print("forecasts replaced by the guard in the new models:", len(caught))
cov = fc9.groupby("model").agg(items=("item_id", "nunique"), origins=("origin", "nunique"))
cov

forecasts replaced by the guard in the new models: 2


,items,origins
model,,
arima,150,9
arma,150,9
autots,150,9
combination,150,9
ets_boxcox,150,9
gru,150,9
hw_add,150,7
lightgbm,150,9
lstm,150,9


## What the tuning picked (validation only)

In [6]:
tune = {}
xp = json.loads((M.CACHE / "xgb_params.json").read_text())
lp = json.loads((M.CACHE / "lgbm_params.json").read_text())
print("XGBoost best settings:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in xp["params"].items()}, "trees:", xp["n_est"])
trials = pd.read_csv(TABLES / "final_xgb_trials.csv")
print(len(trials), "Optuna trials, best validation score", round(trials["value"].min(), 4), "worst", round(trials["value"].max(), 4))
print("LightGBM (reference):", lp["n_est"], "trees")
pt = pd.read_csv(TABLES / "final_prophet_tuning.csv").sort_values("val_score")
print("Prophet grid (12 settings, 30 items, 3 validation folds):")
print(pt.round(4).to_string(index=False))
for k in ["lstm", "gru", "patchtst"]:
    t = pd.read_csv(TABLES / f"final_{k}_tuning.csv").sort_values("val_score")
    print(); print(k, "configs tried:"); print(t.round(4).to_string(index=False))

XGBoost best settings: {'objective': 'reg:pseudohubererror', 'learning_rate': 0.0276, 'max_depth': 8, 'min_child_weight': 1.6848, 'subsample': 0.8984, 'colsample_bytree': 0.521, 'reg_lambda': 2.371} trees: 258
25 Optuna trials, best validation score 6.8989 worst 7.3779
LightGBM (reference): 80 trees
Prophet grid (12 settings, 30 items, 3 validation folds):
 changepoint_prior_scale seasonality_mode  seasonality_prior_scale  val_score
                    0.20         additive                      1.0    10.5707
                    0.01         additive                      1.0    10.5781
                    0.01         additive                     10.0    10.5880
                    0.20         additive                     10.0    10.6464
                    0.05         additive                     10.0    10.7529
                    0.05         additive                      1.0    10.7695
                    0.01   multiplicative                     10.0    11.1616
                 

In [7]:
vr = FE.val_rank(fc9, FE.candidates(fc9))
vr.to_csv(TABLES / "final_validation.csv")
written.append(TABLES / "final_validation.csv")
vr.round(3)

,pooled_mae,pooled_rmse,pooled_mape_nonzero,pooled_wape,bias,rank
model,,,,,,
xgboost,5.111,8.669,0.602,0.371,-0.338,1.0
lightgbm,5.307,8.674,0.699,0.386,0.670,2.0
sarimax,5.398,9.079,0.667,0.392,0.624,3.0
sarima,5.479,9.169,0.689,0.398,0.849,4.0
lstm,5.673,9.254,0.703,0.412,1.140,6.5
ets_boxcox,5.515,9.758,0.633,0.401,0.582,7.0
patchtst,5.702,9.362,0.694,0.414,0.637,7.5
autots,5.663,9.971,0.686,0.411,1.197,8.0
prophet,5.802,9.426,0.738,0.421,1.243,8.5


## Combinations

All three are built from validation forecasts only.

1. **Average of the top three**: the plain mean of the three single models with the best validation rank (mean of the MAE rank
   and the RMSE rank).
2. **Stacked**: non-negative least squares on the validation forecasts of all candidate models. The weights are fixed
   afterwards and applied to the test windows.
3. **Hybrid**: the classical model (SARIMA, Prophet or tuned Prophet, whichever is best on validation) plus an XGBoost model on its
   residuals. For each test window the residual model is fitted on the earlier windows only. There are few windows (3 to 8), so
   its settings are fixed and strongly regularised rather than tuned.

In [8]:
cands = FE.candidates(fc9)
avg, avg_members = FE.average_top(fc9, cands, 3)
stk, weights = FE.stack(fc9, cands)
base = FE.val_rank(fc9, [m for m in ["sarima", "prophet", "prophet_tuned"] if m in cands]).index[0]
hyb = FE.hybrid(fc9, rows, base)
print("candidates:", cands)
print("average of:", avg_members)
print("hybrid base:", base)
w = weights.rename("weight").to_frame()
w["name"] = [FE.NAMES[m] for m in w.index]
w.to_csv(TABLES / "final_stack_weights.csv")
written.append(TABLES / "final_stack_weights.csv")
w.round(4).sort_values("weight", ascending=False)

candidates: ['autots', 'ets_boxcox', 'gru', 'lightgbm', 'lstm', 'patchtst', 'prophet', 'prophet_tuned', 'sarima', 'sarimax', 'seasonal_naive', 'theta', 'xgboost']
average of: ['xgboost', 'lightgbm', 'sarimax']
hybrid base: sarima


,weight,name
xgboost,0.3157,XGBoost (Optuna)
lightgbm,0.2690,LightGBM (reference)
prophet_tuned,0.2087,"Prophet (tuned, events)"
sarimax,0.1355,SARIMAX
lstm,0.0253,LSTM
ets_boxcox,0.0000,"Holt-Winters (damped, Box-Cox)"
gru,0.0000,GRU
autots,0.0000,AutoTS
patchtst,0.0000,PatchTST
sarima,0.0000,SARIMA


## VAR on the category totals

Three related series: the daily totals of FOODS, HOBBIES and HOUSEHOLD (the 150 items added up). Each origin: ADF test for each
series (differenced until stationary at 5%), lag chosen by AIC up to 21, weekday dummies, SNAP and events as regressors. To put VAR
on the same item-level table as the others, its category forecast is shared out to the items by each item's share of the
category over the last 56 days (a top-down split).

In [9]:
var_tot, var_sets, var_items = FE.var_study(daily, inp["cal"], items)
var_sets.to_csv(TABLES / "final_var_settings.csv", index=False)
written.append(TABLES / "final_var_settings.csv")
var_sets

,origin,lag_aic,differences,adf_d_FOODS,adf_d_HOBBIES,adf_d_HOUSEHOLD
0,2015-09-13,14,1,1,1,1
1,2015-10-11,14,1,1,0,1
2,2015-11-08,14,1,0,1,1
3,2015-12-06,13,1,0,1,1
4,2016-01-03,13,1,0,1,1
5,2016-01-31,14,1,1,1,1
6,2016-02-28,14,1,1,1,1
7,2016-03-27,14,1,0,1,1
8,2016-04-24,14,1,0,1,1


## The leaderboard (test windows, scored once)

Headline numbers keep the convention of the earlier notebooks: each item's mean error over the six windows, then the median over
the 150 items. Pooled numbers (all item-days together) sit next to them. MAPE only uses days with sales above zero, and
WAPE (total absolute error over total sales) is the zero-safe check. Ranking uses MAE and RMSE: the rank column is the mean of
the MAE rank and the RMSE rank, ties broken by RMSE. For the neural nets the number is the mean over three seeds and the
spread is the standard deviation over seeds.

In [10]:
singles = pd.concat([fc9, var_items], ignore_index=True)
allfc = pd.concat([singles, avg, stk, hyb], ignore_index=True)
allfc["category"] = allfc["item_id"].str.split("_").str[0]
scales = mase_scales(daily, tm.ALL_ORIGINS)
test_models = [m for m in FE.NAMES if m in set(allfc["model"])]
pi = FE.per_item(allfc[allfc["model"].isin(test_models)], scales)
med = pi.groupby("model")[["mae", "rmse", "mape", "wape", "mase"]].median()
pool = FE.pooled(allfc[allfc["model"].isin(test_models)])

# neural nets: score every seed on its own
sd = {}
for m, g in seeds.groupby("model"):
    parts = []
    for s, gs in g.groupby("seed"):
        pis = FE.per_item(gs.assign(model=f"{m}_{s}"), scales)
        pl = FE.pooled(gs.assign(model=f"{m}_{s}"))
        r = pis.groupby("model")[["mae", "rmse", "mape", "wape", "mase"]].median()
        r = r.join(pl.rename(columns={"pooled_mae": "pooled_mae"}))
        parts.append(r.iloc[0])
    sd[m] = pd.DataFrame(parts)

lb = med.join(pool)
lb.columns = [c if c.startswith("pooled") or c == "bias" else c for c in lb.columns]
lb = lb.rename(columns={"mape": "mape_nonzero"})
for c in ["mae_std", "rmse_std"]:
    lb[c] = np.nan
for m, d in sd.items():
    for c in ["mae", "rmse", "mape", "wape", "mase", "pooled_mae", "pooled_rmse", "pooled_mape_nonzero", "pooled_wape", "bias"]:
        col = "mape_nonzero" if c == "mape" else c
        lb.loc[m, col] = d[c].mean()
    lb.loc[m, "mae_std"], lb.loc[m, "rmse_std"] = d["mae"].std(ddof=1), d["rmse"].std(ddof=1)
lb = lb.reset_index().rename(columns={"index": "model"})
lb["name"] = lb["model"].map(FE.NAMES)
lb["family"] = lb["model"].map(FE.FAMILY)
lb["level"] = "item"
lb["rank_mae"] = lb["mae"].rank(method="min")
lb["rank_rmse"] = lb["rmse"].rank(method="min")
lb["score"] = (lb["rank_mae"] + lb["rank_rmse"]) / 2
lb = lb.sort_values(["score", "rmse"]).reset_index(drop=True)
lb["rank"] = np.arange(1, len(lb) + 1)
lb[["model", "name", "family", "mae", "mae_std", "rmse", "rmse_std", "mape_nonzero", "wape", "rank"]].round(3)

,model,name,family,mae,mae_std,rmse,rmse_std,mape_nonzero,wape,rank
0,xgboost,XGBoost (Optuna),Machine learning,3.405,NaN,4.351,NaN,0.557,0.425,1
1,avg_top3,Average of top 3,Combination,3.487,NaN,4.378,NaN,0.587,0.428,2
2,stack,Stacked (non-negative weights),Combination,3.460,NaN,4.421,NaN,0.566,0.430,3
3,combination,"Combination (SARIMAX, HW, Theta)",Combination,3.499,NaN,4.392,NaN,0.586,0.437,4
4,hybrid,Hybrid (classical + XGBoost residuals),Combination,3.496,NaN,4.445,NaN,0.587,0.427,5
5,sarima,SARIMA,Classical,3.522,NaN,4.445,NaN,0.603,0.435,6
6,ets_boxcox,"Holt-Winters (damped, Box-Cox)",Classical,3.543,NaN,4.513,NaN,0.560,0.434,7
7,theta,Theta,Classical,3.510,NaN,4.541,NaN,0.602,0.442,8
8,lightgbm,LightGBM (reference),Machine learning,3.590,NaN,4.516,NaN,0.623,0.439,9
9,arma,"ARMA(p, q)",Classical,3.577,NaN,4.542,NaN,0.617,0.454,10


### Training time

Seconds to fit on the six test windows for all 150 items. The new models were timed while they ran (XGBoost and LightGBM: six
refits; Prophet: the sum of the item fits; neural nets: one training run per seed, mean over seeds). The earlier models were not
timed when they were first run, so they get a short timing run now, scaled up to 150 items and six windows; it is an estimate,
and AutoTS, ARIMA and ARMA are left out. VAR is timed on its six fits. Tuning time is not included.

In [11]:
tfile = PROCESSED / "final" / "times_classical.json"
if tfile.exists():
    old_t = json.loads(tfile.read_text())
else:
    old_t = FE.time_classical(inp)
    old_t["var"] = FE.time_var(daily, inp["cal"], items)
    tfile.write_text(json.dumps(old_t))
new_t = FE.tune_seconds()
lb["fit_seconds"] = lb["model"].map({**old_t, **new_t})
lb["fit_seconds_kind"] = lb["model"].map(lambda m: "measured" if m in new_t else ("estimated" if m in old_t else ""))
# combinations: the time of their parts plus the small fitting step
parts = {"avg_top3": avg_members, "stack": [m for m in weights.index if weights[m] > 1e-6], "hybrid": [base, "xgboost"]}
for k, ms in parts.items():
    t = [lb.loc[lb["model"] == m, "fit_seconds"].iloc[0] for m in ms]
    lb.loc[lb["model"] == k, "fit_seconds"] = np.nansum(t) if not any(np.isnan(t)) else np.nan
    lb.loc[lb["model"] == k, "fit_seconds_kind"] = "sum of its models"
lb[["model", "name", "fit_seconds", "fit_seconds_kind"]].round(1)

,model,name,fit_seconds,fit_seconds_kind
0,xgboost,XGBoost (Optuna),168.7,measured
1,avg_top3,Average of top 3,1139.4,sum of its models
2,stack,Stacked (non-negative weights),3505.3,sum of its models
3,combination,"Combination (SARIMAX, HW, Theta)",1095.7,estimated
4,hybrid,Hybrid (classical + XGBoost residuals),707.7,sum of its models
5,sarima,SARIMA,539.0,estimated
6,ets_boxcox,"Holt-Winters (damped, Box-Cox)",200.8,estimated
7,theta,Theta,10.4,estimated
8,lightgbm,LightGBM (reference),86.2,measured
9,arma,"ARMA(p, q)",NaN,


In [12]:
best_row = lb.iloc[0]
BEST = best_row["model"]
best_single = lb[lb["family"] != "Combination"].iloc[0]["model"]
print("best overall:", FE.NAMES[BEST], "| best single model:", FE.NAMES[best_single])
combos = lb[lb["family"] == "Combination"]
singles_lb = lb[lb["family"] != "Combination"]
bs = singles_lb.iloc[0]
for _, r in combos.iterrows():
    better_mae, better_rmse = r["mae"] < bs["mae"], r["rmse"] < bs["rmse"]
    print(f"{r['name']}: MAE {r['mae']:.3f}, RMSE {r['rmse']:.3f} vs best single {bs['name']} MAE {bs['mae']:.3f}, RMSE {bs['rmse']:.3f} -> "
          f"{'beats it on both' if better_mae and better_rmse else 'beats it on MAE only' if better_mae else 'beats it on RMSE only' if better_rmse else 'does not beat it'}")
lb.to_csv(TABLES / "final_leaderboard.csv", index=False)
(TABLES / "final_leaderboard.json").write_text(json.dumps(json.loads(lb.to_json(orient="records")), indent=1))
written += [TABLES / "final_leaderboard.csv", TABLES / "final_leaderboard.json"]
lb[["rank", "name", "family", "mae", "rmse", "mape_nonzero", "wape", "pooled_mae", "pooled_rmse", "fit_seconds"]].round(3)

best overall: XGBoost (Optuna) | best single model: XGBoost (Optuna)
Average of top 3: MAE 3.487, RMSE 4.378 vs best single XGBoost (Optuna) MAE 3.405, RMSE 4.351 -> does not beat it
Stacked (non-negative weights): MAE 3.460, RMSE 4.421 vs best single XGBoost (Optuna) MAE 3.405, RMSE 4.351 -> does not beat it
Combination (SARIMAX, HW, Theta): MAE 3.499, RMSE 4.392 vs best single XGBoost (Optuna) MAE 3.405, RMSE 4.351 -> does not beat it
Hybrid (classical + XGBoost residuals): MAE 3.496, RMSE 4.445 vs best single XGBoost (Optuna) MAE 3.405, RMSE 4.351 -> does not beat it


,rank,name,family,mae,rmse,mape_nonzero,wape,pooled_mae,pooled_rmse,fit_seconds
0,1,XGBoost (Optuna),Machine learning,3.405,4.351,0.557,0.425,5.018,8.554,168.711
1,2,Average of top 3,Combination,3.487,4.378,0.587,0.428,5.019,8.353,1139.414
2,3,Stacked (non-negative weights),Combination,3.460,4.421,0.566,0.430,5.015,8.431,3505.342
3,4,"Combination (SARIMAX, HW, Theta)",Combination,3.499,4.392,0.586,0.437,5.125,8.635,1095.706
4,5,Hybrid (classical + XGBoost residuals),Combination,3.496,4.445,0.587,0.427,5.238,8.870,707.691
5,6,SARIMA,Classical,3.522,4.445,0.603,0.435,5.205,8.644,538.979
6,7,"Holt-Winters (damped, Box-Cox)",Classical,3.543,4.513,0.560,0.434,5.211,9.016,200.794
7,8,Theta,Classical,3.510,4.541,0.602,0.442,5.259,8.904,10.445
8,9,LightGBM (reference),Machine learning,3.590,4.516,0.623,0.439,5.123,8.468,86.235
9,10,"ARMA(p, q)",Classical,3.577,4.542,0.617,0.454,5.315,8.703,NaN


The VAR is also scored at the category totals, where it is natural. Here every item-level model is summed to the category and
compared on the same days. These are bigger numbers (units a day for a whole category), so they are not comparable with the
item table above.

In [13]:
cat_of = {i: i.split("_")[0] for i in items}
tot_models = [m for m in ["seasonal_naive", "sarima", "combination", BEST, "xgboost", "lstm", "gru", "patchtst"] if m in set(allfc["model"])]
tf = FE.totals_of(allfc[allfc["model"].isin(tot_models) & allfc["origin"].isin(EVAL_ORIGINS)], cat_of)
tf = pd.concat([tf, var_tot[var_tot["origin"].isin(EVAL_ORIGINS)][["series", "origin", "date", "model", "forecast", "actual"]]])
from metrics import mae, rmse, wape
tt = tf.groupby("model").apply(lambda g: pd.Series({"mae": mae(g["actual"], g["forecast"]), "rmse": rmse(g["actual"], g["forecast"]),
                                                    "wape": wape(g["actual"], g["forecast"])}), include_groups=False)
tt["name"] = [FE.NAMES[m] for m in tt.index]
tt["level"] = "category totals (3 series)"
tt = tt.sort_values("rmse")
tt.to_csv(TABLES / "final_totals_level.csv")
written.append(TABLES / "final_totals_level.csv")
tt.round(3)

,mae,rmse,wape,name,level
model,,,,,
xgboost,74.384,103.089,0.109,XGBoost (Optuna),category totals (3 series)
combination,70.727,107.283,0.103,"Combination (SARIMAX, HW, Theta)",category totals (3 series)
sarima,73.241,108.467,0.107,SARIMA,category totals (3 series)
var,75.431,111.411,0.110,VAR (top-down to items),category totals (3 series)
seasonal_naive,89.149,128.933,0.130,Seasonal naive,category totals (3 series)
gru,106.320,147.173,0.156,GRU,category totals (3 series)
patchtst,132.961,176.182,0.195,PatchTST,category totals (3 series)
lstm,153.311,205.369,0.224,LSTM,category totals (3 series)


In [14]:
vrow = tt.loc["var"]
lb = pd.read_csv(TABLES / "final_leaderboard.csv")
lb.loc[lb["model"] == "var", "note"] = f"item level is a top-down split; at category totals VAR has MAE {vrow['mae']:.1f}, RMSE {vrow['rmse']:.1f}"
lb.to_csv(TABLES / "final_leaderboard.csv", index=False)
(TABLES / "final_leaderboard.json").write_text(json.dumps(json.loads(lb.to_json(orient="records")), indent=1))
print(lb.loc[lb["model"] == "var", "note"].iloc[0])

item level is a top-down split; at category totals VAR has MAE 75.4, RMSE 111.4


## Is the best model really better? Diebold-Mariano

Loss differences per day (averaged over the 150 items) over the 168 test days, Newey-West variance with 27 lags, and the
Harvey-Leybourne-Newbold correction for the 28-day horizon. Absolute-error and squared-error loss each get a test. Negative
statistic: the first model is better. Paired Wilcoxon tests per item (as in the earlier notebooks) are added as a second view.
Neural nets here use the seed-averaged forecast.

In [15]:
best_classical = lb[lb["model"].isin(FE.CLASSICAL + ["combination"])].iloc[0]["model"]
pairs = [(BEST, "seasonal_naive"), (BEST, "sarima"), (BEST, best_classical if best_classical != BEST else "sarima"),
         (BEST, best_single if best_single != BEST else "xgboost")]
pairs = [p for i, p in enumerate(pairs) if p[0] != p[1] and p not in pairs[:i]]
test_fc = allfc[allfc["origin"].isin(EVAL_ORIGINS)]
dm = VM.dm_table(test_fc, pairs)
from metrics import paired_test
w = []
for a, b in pairs:
    x = pi[pi["model"] == a].set_index("item_id")["mae"] - pi[pi["model"] == b].set_index("item_id")["mae"]
    t = paired_test(x)
    w.append({"model_a": a, "model_b": b, "median_item_mae_diff": t["median_diff"], "share_items_a_better": t["share_improved"], "wilcoxon_p": t["p_value"]})
dm = dm.merge(pd.DataFrame(w), on=["model_a", "model_b"])
dm.to_csv(TABLES / "final_dm_tests.csv", index=False)
written.append(TABLES / "final_dm_tests.csv")
print("best:", FE.NAMES[BEST], "| best classical-family model:", FE.NAMES[best_classical])
dm.round(4)

best: XGBoost (Optuna) | best classical-family model: Combination (SARIMAX, HW, Theta)


,model_a,model_b,loss,days,mean_loss_a,mean_loss_b,dm_stat,p_value,a_better,significant_5pct,median_item_mae_diff,share_items_a_better,wilcoxon_p
0,xgboost,seasonal_naive,absolute error,168,5.0177,6.5482,-10.0596,0.0000,True,True,-1.2255,0.9867,0.0
1,xgboost,seasonal_naive,squared error,168,73.1650,116.2636,-6.0958,0.0000,True,True,-1.2255,0.9867,0.0
2,xgboost,sarima,absolute error,168,5.0177,5.2045,-1.9574,0.0520,True,False,-0.1088,0.7733,0.0
3,xgboost,sarima,squared error,168,73.1650,74.7274,-0.4443,0.6574,True,False,-0.1088,0.7733,0.0
4,xgboost,combination,absolute error,168,5.0177,5.1249,-1.2871,0.1998,True,False,-0.0538,0.7400,0.0
5,xgboost,combination,squared error,168,73.1650,74.5640,-0.6059,0.5454,True,False,-0.0538,0.7400,0.0


## Error by horizon, and where the models fail

In [16]:
top = list(lb["model"].head(4))
hmods = list(dict.fromkeys(top + ["seasonal_naive", "sarima"]))
hz = FE.horizon_table(allfc, [m for m in lb["model"]])
hz.to_csv(TABLES / "final_horizon.csv", index=False)
written.append(TABLES / "final_horizon.csv")
brk = pd.read_csv(PROCESSED / "breaks_by_origin.csv", parse_dates=["origin"])
seg = FE.segment_table(allfc, [m for m in lb["model"]], inp["cal"], daily, brk)
seg.to_csv(TABLES / "final_segments.csv", index=False)
written.append(TABLES / "final_segments.csv")
s = seg[seg["model"].isin([BEST, "seasonal_naive", "sarima"])].pivot(index="segment", columns="model", values="mae")
s["best / naive"] = s[BEST] / s["seasonal_naive"] if BEST != "seasonal_naive" else np.nan
s.join(seg[seg["model"] == BEST].set_index("segment")[["item_days", "bias"]]).round(3)

,sarima,seasonal_naive,xgboost,best / naive,item_days,bias
segment,,,,,,
SNAP days,5.038,6.399,4.991,0.780,8100,-0.847
all days,5.205,6.548,5.018,0.766,25200,-1.078
days with zero sales,7.250,5.221,4.641,0.889,1360,4.641
event days,5.372,6.945,5.227,0.753,2400,-1.110
items with a detected shift,5.028,6.402,4.911,0.767,12600,-0.953
items without a shift,5.381,6.695,5.124,0.765,12600,-1.204
no event,5.187,6.506,4.996,0.768,22800,-1.075
non-SNAP days,5.283,6.619,5.030,0.760,17100,-1.187
weekday,4.946,6.171,4.692,0.760,18000,-0.963


## GARCH on the residuals of the best model

GARCH(1,1) with a zero mean, fitted to the best model's forecast residuals (put on one scale by dividing by the item's usual level).
It models how big the errors are from day to day. It does not change the forecast, so MAE and RMSE stay exactly the same. What it
can change is the prediction interval. For each test window the model is fitted on the residuals available before that window
(the validation windows and earlier test windows) and its variance forecast gives a band around the forecast, compared with a
constant-spread band. The Engle ARCH-LM test says if the residuals show changing variance at all.

In [17]:
fb = allfc[(allfc["model"] == BEST)]
assert fb["origin"].nunique() == len(tm.ALL_ORIGINS), "the best model needs forecasts at all nine origins for the GARCH step"
iv, gsum, lm, daytype = FE.garch_study(fb, daily, inp["cal"], items)
iv.to_parquet(TABLES / "final_intervals.parquet")
gsum.to_csv(TABLES / "final_garch_intervals.csv", index=False)
lm.to_csv(TABLES / "final_garch_archlm.csv", index=False)
daytype.to_csv(TABLES / "final_garch_daytype.csv", index=False)
written += [TABLES / "final_intervals.parquet", TABLES / "final_garch_intervals.csv", TABLES / "final_garch_archlm.csv", TABLES / "final_garch_daytype.csv"]
print(f"GARCH converged for {iv['garch_ok'].mean():.0%} of item-windows; ARCH-LM rejects constant variance (5%) for {(lm['arch_lm_p'] < 0.05).mean():.0%} of items")
print("MAE of the forecast is unchanged by construction:", round(mae(iv["actual"], iv["forecast"]), 4))
display(gsum.round(3))
daytype.round(3)

GARCH converged for 100% of item-windows; ARCH-LM rejects constant variance (5%) for 35% of items
MAE of the forecast is unchanged by construction: 5.0177


,nominal,interval,coverage,mean_width,interval_score
0,0.80,constant std,0.824,16.556,24.298
1,0.80,"GARCH(1,1)",0.828,17.230,24.978
2,0.95,constant std,0.939,23.667,36.574
3,0.95,"GARCH(1,1)",0.939,24.474,38.503


,day_type,item_days,mean_sigma_over_scale,mean_abs_error_over_scale
0,event day,2400,0.596,0.457
1,no event,22800,0.595,0.437
2,SNAP day,8100,0.591,0.440
3,non-SNAP day,17100,0.597,0.439
4,weekend,7200,0.597,0.503
5,weekday,18000,0.595,0.414


## Inventory with the final forecasts

The stock simulation of 11e again (7-day lead time, 95% service level, same costs, both the normal-formula and the calibrated
safety stock), with the new best model next to seasonal naive, SARIMA and the earlier best (the combination). Costs are assumptions, as before.

In [18]:
long_df = inp["long_df"]
inv_models = [m for m in dict.fromkeys(["seasonal_naive", "sarima", "combination", BEST, best_single]) if m in set(allfc["model"])]
warm = allfc[allfc["origin"] == ORIGINS[0]]["model"].unique()
inv_models = [m for m in inv_models if m in warm]
st_items, st_sum = FE.inventory_rerun(allfc, inv_models, long_df, inp["cal"], items)
st_items.to_csv(TABLES / "final_stock_items.csv", index=False)
st_sum.to_csv(TABLES / "final_stock_summary.csv", index=False)
written += [TABLES / "final_stock_items.csv", TABLES / "final_stock_summary.csv"]
st_sum[["method", "model", "fill_rate", "cycle_service_level", "avg_inventory_value", "holding_cost", "stockout_cost", "total_cost"]].round(4)

,method,model,fill_rate,cycle_service_level,avg_inventory_value,holding_cost,stockout_cost,total_cost
0,conformal,combination,0.9900,0.8992,37962.0476,4368.2356,3892.5845,8260.8201
1,conformal,sarima,0.9915,0.9045,36983.0628,4255.5853,3600.9615,7856.5468
2,conformal,seasonal_naive,0.9887,0.8861,40272.0058,4634.0390,4339.5048,8973.5439
3,conformal,xgboost,0.9910,0.9223,36701.2595,4223.1586,3616.4886,7839.6472
4,normal formula,combination,0.9796,0.8447,33410.6984,3844.5187,7389.8569,11234.3756
5,normal formula,sarima,0.9831,0.8575,32866.0845,3781.8508,6849.5186,10631.3694
6,normal formula,seasonal_naive,0.9809,0.8668,37897.0822,4360.7601,7076.2975,11437.0576
7,normal formula,xgboost,0.9742,0.8095,29202.0146,3360.2318,9672.6791,13032.9109


In [19]:
ref = "combination"
rows_ = []
rng = np.random.default_rng(SEED)
for method, g in st_items.groupby("method"):
    cost = g.pivot(index="item_id", columns="model", values="total_cost")
    stock = g.pivot(index="item_id", columns="model", values="avg_inventory_value")
    for b in [m for m in inv_models if m != BEST]:
        d = (cost[BEST] - cost[b]).to_numpy()
        boot = d[rng.integers(0, len(d), size=(4000, len(d)))].sum(axis=1)
        rows_.append({"method": method, "model": BEST, "versus": b, "total_cost_diff": d.sum(), "total_cost_diff_pct": d.sum() / cost[b].sum(),
                      "ci_low": np.quantile(boot, 0.025), "ci_high": np.quantile(boot, 0.975),
                      "stock_value_diff_pct": stock[BEST].sum() / stock[b].sum() - 1})
stock_cmp = pd.DataFrame(rows_)
stock_cmp.to_csv(TABLES / "final_stock_compare.csv", index=False)
written.append(TABLES / "final_stock_compare.csv")
stock_cmp.round(4)

,method,model,versus,total_cost_diff,total_cost_diff_pct,ci_low,ci_high,stock_value_diff_pct
0,conformal,xgboost,seasonal_naive,-1133.8966,-0.1264,-2728.1052,144.3782,-0.0887
1,conformal,xgboost,sarima,-16.8996,-0.0022,-714.0794,742.2975,-0.0076
2,conformal,xgboost,combination,-421.1729,-0.0510,-1611.9893,678.8469,-0.0332
3,normal formula,xgboost,seasonal_naive,1595.8533,0.1395,-680.3173,4117.6321,-0.2294
4,normal formula,xgboost,sarima,2401.5415,0.2259,733.5246,4503.2815,-0.1115
5,normal formula,xgboost,combination,1798.5353,0.1601,-223.7879,4066.1929,-0.1260


## Figures

In [20]:
FP.bars(lb, written)
ctop = [BEST] + [m for m in lb["model"] if m != BEST][:3]
mid = inp["sample"].sort_values("mean_sales") if "mean_sales" in inp["sample"] else inp["sample"]
sel = []
for c in ["FOODS", "HOBBIES", "HOUSEHOLD"]:
    ids = [i for i in items if i.startswith(c)]
    tot_s = {i: daily[i]["sales"].iloc[-365:].mean() for i in ids}
    sel.append(sorted(ids, key=tot_s.get)[len(ids) // 2])
FP.overlay(allfc[allfc["origin"].isin(EVAL_ORIGINS)], ctop + ["seasonal_naive"], sel, written)
FP.horizon(hz, list(dict.fromkeys(ctop + ["seasonal_naive"])), written)
FP.residuals(allfc, BEST, written)
FP.weights(weights, avg_members, written)
FP.segments(seg, ctop[:3], written)
FP.volatility(daytype, iv, sel[0], written)
FP.stock(st_sum, inv_models, written)
for p in written:
    if p.suffix == ".png":
        print(p.name)

final_mae_rmse_bars.png
final_forecast_overlay.png
final_error_by_horizon.png
final_residuals.png
final_ensemble_weights.png
final_failure_segments.png
final_garch_volatility.png
final_stock_impact.png


## What the numbers say

The sentences below are written by `src/final_text.py` from the tables above, and the report, the slides and the dashboard use
the same ones, so the numbers cannot drift apart. They are the answer to the questions of this notebook: which model wins, whether a
combination wins, whether the gain is real, where the models fail, what GARCH shows and what it means for stock.

In [21]:
import textwrap
import final_text
tx = final_text.Final().all()
for k, v in tx.items():
    print("##", k)
    for p in (v if isinstance(v, list) else [v]):
        print(textwrap.fill(p, 130)); print()

## winner
XGBoost (Optuna) is first on both ranking measures: median MAE 3.41 and RMSE 4.35 units a day, 28% and 28% below seasonal naive
(4.72 and 6.04). The best earlier model, Combination (SARIMAX, HW, Theta), scores 3.50 and 4.39. The ranking depends a little on
how errors are pooled. With all item-days together, Stacked (non-negative weights) has the lowest MAE (5.015) and Average of top 3
the lowest RMSE (8.353), against 5.018 and 8.554 for XGBoost (Optuna). The gaps among the top models are small.

## combos
No combination beat the best single model, XGBoost (Optuna), on both MAE and RMSE, so the hypothesis that a combination wins is
not supported by these results. Average of top 3: MAE 3.49, RMSE 4.38 (not better than XGBoost (Optuna)'s 3.41 and 4.35). Stacked
(non-negative weights): MAE 3.46, RMSE 4.42 (not better than XGBoost (Optuna)'s 3.41 and 4.35). Combination (SARIMAX, HW, Theta):
MAE 3.50, RMSE 4.39 (not better than XGBoost (Optuna)'s 3.41 and 4.35). Hybrid (classical +

## Saved forecasts for the dashboard

In [22]:
keep = allfc[allfc["origin"].isin(tm.ALL_ORIGINS)][["item_id", "origin", "date", "model", "forecast", "actual"]].copy()
keep["split"] = np.where(keep["origin"].isin(EVAL_ORIGINS), "test", "validation")
keep.to_parquet(TABLES / "final_forecasts.parquet")
written.append(TABLES / "final_forecasts.parquet")
seeds.drop(columns=["actual"]).to_parquet(TABLES / "final_nn_seed_forecasts.parquet")
print(keep.shape, keep["model"].nunique(), "models")

(772800, 7) 21 models


In [23]:
# the best overall model, for the files the report and dashboard read
info = {"model": BEST, "name": FE.NAMES[BEST], "best_single": best_single, "best_single_name": FE.NAMES[best_single],
        "avg_top3_members": avg_members, "hybrid_base": base, "stack_weights": {k: float(v) for k, v in weights.items() if v > 1e-6}}
(TABLES / "final_best_model.json").write_text(json.dumps(info, indent=1))
written.append(TABLES / "final_best_model.json")
for p in written:
    print(p.relative_to(TABLES.parents[1]), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\final_split.csv 0 KB
outputs\tables\final_features.csv 1 KB
outputs\tables\final_validation.csv 1 KB
outputs\tables\final_stack_weights.csv 0 KB
outputs\tables\final_var_settings.csv 0 KB
outputs\tables\final_leaderboard.csv 6 KB
outputs\tables\final_leaderboard.json 13 KB
outputs\tables\final_totals_level.csv 1 KB
outputs\tables\final_dm_tests.csv 1 KB
outputs\tables\final_horizon.csv 28 KB
outputs\tables\final_segments.csv 24 KB
outputs\tables\final_intervals.parquet 659 KB
outputs\tables\final_garch_intervals.csv 0 KB
outputs\tables\final_garch_archlm.csv 6 KB
outputs\tables\final_garch_daytype.csv 0 KB
outputs\tables\final_stock_items.csv 299 KB
outputs\tables\final_stock_summary.csv 2 KB
outputs\tables\final_stock_compare.csv 1 KB
outputs\figures\final_mae_rmse_bars.png 152 KB
outputs\figures\final_forecast_overlay.png 479 KB
outputs\figures\final_error_by_horizon.png 209 KB
outputs\figures\final_residuals.png 168 KB
outputs\figures\final_ensemble_weights.png 58 KB
